# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [4]:
# Add code here 🔧
url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/airbnb_listings.csv'
df = pd.read_csv(url)
df.head()


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,host_url,host_name,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20250804133828,2025-08-04,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://www.airbnb.com/users/show/4621559,Kenneth,...,4.56,3.22,3.67,NaN,0,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20250804133828,2025-08-04,city scrape,Funky Urban Gem: Prime Central Location - Park...,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.81,4.81,4.77,NaN,0,4,4,0,0,2.32
2,5651579,https://www.airbnb.com/rooms/5651579,20250804133828,2025-08-04,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://www.airbnb.com/users/show/29288920,Gregg,...,4.88,4.76,4.64,NaN,0,2,1,1,0,2.97
3,6623339,https://www.airbnb.com/rooms/6623339,20250804133828,2025-08-04,city scrape,Bright & Cozy City Stay · Top Location + Parking!,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.70,4.80,4.72,NaN,0,4,4,0,0,2.68
4,9005989,https://www.airbnb.com/rooms/9005989,20250804133828,2025-08-04,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://www.airbnb.com/users/show/17766924,Sugey,...,4.93,4.87,4.77,NaN,0,1,1,0,0,5.67


In [5]:
df.shape

(459, 77)

### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
1. To name a couple, some of the business info that the dataset includes are review scores and number of listings.

2. There are 459 rows and 77 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [6]:
# Add code here 🔧
df.drop(columns=['id', 'name', 'description', 'details', 'address'], inplace=True, errors='ignore')

### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. I dropped the columns that contained string data because they were aliases and descriptions rather than meaningful data.

2. Although string data isn't as easy to analyze, it may contain some valuable information. For instance, maybe the description is what influences Airbnb guests to book a spot because it shows the effort that the host puts into describing their listing.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [7]:
# Add code here 🔧
correlation_matrix = df.corr(numeric_only=True)

correlation_matrix

,scrape_id,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
scrape_id,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_listings_count,NaN,1.000000,0.973960,-0.070382,0.111195,0.037236,-0.050743,0.041395,0.057712,-0.014677,...,-0.341142,-0.241995,-0.375941,NaN,0.277760,0.015981,0.057724,-0.061563,NaN,-0.116031
host_total_listings_count,NaN,0.973960,1.000000,-0.072907,0.099717,0.057426,-0.044631,0.061755,0.086508,-0.006580,...,-0.302003,-0.212705,-0.304869,NaN,0.252688,0.007070,0.048405,-0.062113,NaN,-0.103788
latitude,NaN,-0.070382,-0.072907,1.000000,-0.548236,0.006482,0.067394,0.016919,0.021012,-0.014666,...,0.101162,0.108243,0.066316,NaN,0.091104,0.120997,-0.017696,0.227452,NaN,-0.119777
longitude,NaN,0.111195,0.099717,-0.548236,1.000000,-0.083466,-0.175493,-0.215899,-0.138303,-0.118913,...,-0.166284,-0.332212,-0.214617,NaN,0.043940,-0.047019,0.109600,-0.245058,NaN,0.045170
accommodates,NaN,0.037236,0.057426,0.006482,-0.083466,1.000000,0.520394,0.809299,0.828875,0.579588,...,0.018571,-0.005875,0.009148,NaN,0.014482,-0.025960,0.100762,-0.196689,NaN,0.075746
bathrooms,NaN,-0.050743,-0.044631,0.067394,-0.175493,0.520394,1.000000,0.534375,0.462908,0.468030,...,0.008527,0.111561,0.098843,NaN,-0.018413,0.020478,-0.058583,0.129067,NaN,-0.000851
bedrooms,NaN,0.041395,0.061755,0.016919,-0.215899,0.809299,0.534375,1.000000,0.784348,0.499286,...,-0.005564,-0.007703,0.016780,NaN,-0.033193,-0.040207,0.018162,-0.094575,NaN,-0.049262
beds,NaN,0.057712,0.086508,0.021012,-0.138303,0.828875,0.462908,0.784348,1.000000,0.547032,...,0.001802,-0.042161,0.000307,NaN,0.014030,-0.135324,-0.057550,-0.140551,NaN,0.078050
price,NaN,-0.014677,-0.006580,-0.014666,-0.118913,0.579588,0.468030,0.499286,0.547032,1.000000,...,-0.131798,-0.015814,0.018269,NaN,-0.013742,0.015773,0.033206,-0.024513,NaN,-0.090843


### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. The variables with the strongest positive correlations with price are accomodates, beds, bedrooms, and bathrooms. The more there are of these variables, the higher the prices.

2. I don't think there are many other useful variables as the rest have very weak positive and negative relationships with price.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [8]:
# Add code here 🔧
y = df['price']
X = df.drop(columns=['price'])

### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. Features that we're using include accomodates, beds, bedrooms, bathrooms along with instant bookable and count of host lstiings.

2. This is a regression problem because we are predicting a numerical value, price, rather than a category.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [9]:
# Add code here 🔧
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [14]:
# Add code here 🔧
y = df['price']
X = df.drop(columns=['price'])

# Keep only numerical features
X = X.select_dtypes(include='number')
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.select_dtypes(include='object').columns)


Index([], dtype='object')


In [17]:
X_train.isnull().sum()[X_train.isnull().sum() > 0]
empty_columns = X_train.columns[X_train.isnull().all()]

X_train = X_train.drop(columns=empty_columns)
X_test = X_test.drop(columns=empty_columns)
X_train = X_train.fillna(X_train.mean())
X_test = X_test.fillna(X_train.mean())
print(X_train.isnull().sum().sum())
print(X_test.isnull().sum().sum())

0
0


In [18]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [20]:
# Predict Prices for Test Set
y_pred = model.predict(X_test)

## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [21]:
# Add code here 🔧
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R²:", r2)

MSE: 8200.752116994545
R²: -0.4611589062295043


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response: 🔧
1. My R^2 is in the negatives, so my model is struggling to explain the variation in price.

2. Similarly, the MSE is not looking too hot with a number of 8,200.75. I could improve the model by adding more listings as right now there's 459. I could also do a sweep for missing important factors that may have a strong relationship with pricing.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [22]:
# Add code here 🔧
coefficients = pd.DataFrame({'Feature': X_train.columns,'Coefficient': model.coef_})
coefficients['Absolute Coefficient'] = coefficients['Coefficient'].abs()
coefficients = coefficients.sort_values(by='Absolute Coefficient', ascending=False)

coefficients

,Feature,Coefficient,Absolute Coefficient
3,latitude,-8.383187e+02,8.383187e+02
4,longitude,-4.136645e+02,4.136645e+02
34,review_scores_value,1.404921e+02,1.404921e+02
32,review_scores_communication,-1.356303e+02,1.356303e+02
31,review_scores_checkin,-9.141651e+01,9.141651e+01
6,bathrooms,3.399028e+01,3.399028e+01
15,minimum_nights_avg_ntm,3.358894e+01,3.358894e+01
12,maximum_minimum_nights,-3.298734e+01,3.298734e+01
30,review_scores_cleanliness,-2.918944e+01,2.918944e+01
29,review_scores_accuracy,2.781373e+01,2.781373e+01


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1. Features that increased price the most were review score value, bathrooms, and minimum nights average.

2. I found the review scores communication surprising. Surely, guests wouldn't choose a listing with an uncommunicative host, right?

3. I can draw from the previous question that there are more significant features that are valued when it comes to pricing than how well the hosts communicate. It's how well the host can perform or how much they can materially offer that matters.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [23]:
# Add code here 🔧
top_features = [
    'latitude',
    'longitude',
    'review_scores_value',
    'review_scores_communication',
    'review_scores_checkin']

X_train_top = X_train[top_features]
X_test_top = X_test[top_features]
refined_model = LinearRegression()
refined_model.fit(X_train_top, y_train)
y_pred_top = refined_model.predict(X_test_top)

mse_top = mean_squared_error(y_test, y_pred_top)
r2_top = r2_score(y_test, y_pred_top)
print("Refined MSE:", mse_top)
print("Refined R²:", r2_top)

Refined MSE: 6313.108816713606
Refined R²: -0.12483038652285461


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

### ✍️ Your Response: 🔧
1. I kept latitude, longitude, review scores value, review scores communication, and review scores checkin as they had the strongest correlations.

2. Model performance did improve. The MSE lowered from 8200 to 6313, and the R^2 increased from -0.46 to -0.12.

3. I would recommend the refined model as it has a better MSE and R^2 and not an overwhelming number of features.

4. This related to my customized learning outcome because I'm drawing insights from regression evaluation metrics like MSE and R^2 and making inferences from the regression coefficients. I'm being challenged to think critically and creatively, which is important for someone studying Ops & Supply Chain.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1.  This model helped me answer what features had the most significant influence on price.

2. I'd recommend hosts to take good care of their listings and guests then encourage or incentivize their guests to leave a review rating their stay. Hopefully the rating is high and the host can increase the pricing, so they can earn more.

3. Next time, I would add more listings. I think I would also add a few more positively correlated features and remove the negatively related features.

4. As mentioned before, this related to my customer learning outcome because I'm challenging myself to think independently and analytically, which is an important skill in my field of study.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [24]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"

[NbConvertApp] WARNING | pattern 'assignment_11_regression.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
